In [9]:
from pathlib import Path

import mne
import numpy as np
import pyxdf

# 1) Choose folder + input type first
input_dir = Path(r"./data/2-back/eeg")
input_type = "fif"  # "xdf" or "fif"
scale_to_volts = 1e-6

if input_type not in {"xdf", "fif"}:
    raise ValueError("input_type must be 'xdf' or 'fif'")


def pick_eeg_stream(streams):
    # Keep it simple: pick first stream typed as EEG
    return next(s for s in streams if str(s["info"]["type"][0]).lower() == "eeg")


def xdf_to_raw(xdf_path):
    streams, _ = pyxdf.load_xdf(str(xdf_path))
    eeg_stream = pick_eeg_stream(streams)

    data = np.asarray(eeg_stream["time_series"], dtype=float).T * scale_to_volts
    sfreq = float(eeg_stream["info"]["nominal_srate"][0])
    ch_names = [f"EEG{i+1:03d}" for i in range(data.shape[0])]

    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types="eeg")
    return mne.io.RawArray(data, info)


# 2) Collect files by selected type
pattern = "*.xdf" if input_type == "xdf" else "*.fif"
files = sorted(input_dir.rglob(pattern))
print("Mode:", input_type)
print("Found", len(files), "file(s)")
for f in files:
    print(" -", f)

if not files:
    raise FileNotFoundError(f"No {input_type.upper()} files found in: {input_dir}")

# 3) Load raws
raws = []
for path in files:
    if input_type == "xdf":
        raw = xdf_to_raw(path)
    else:
        raw = mne.io.read_raw_fif(path, preload=True)

    raws.append(raw)
    print(f"Loaded {path.name} | {raw.info['nchan']} ch | {raw.n_times / raw.info['sfreq']:.1f} s")

# 4) Keep only compatible raws, then concatenate
base_names = raws[0].ch_names
base_sfreq = raws[0].info["sfreq"]
aligned = [r for r in raws if r.ch_names == base_names and r.info["sfreq"] == base_sfreq]

if len(aligned) != len(raws):
    print(f"Warning: {len(raws) - len(aligned)} file(s) skipped due to channel/sfreq mismatch")

raw_concat = mne.concatenate_raws(aligned)
print("\nConcatenated raw:")
print(raw_concat)
print("Duration (s):", round(raw_concat.n_times / raw_concat.info["sfreq"], 2))

# Optional save
out_path = input_dir / f"concatenated_{input_type}.fif"
raw_concat.save(out_path, overwrite=True)

Mode: fif
Found 2 file(s)
 - data\2-back\eeg\sub-P001\ses-S001\eeg\sub-P001_ses-S001_task-Default_run-001_eeg_annot_raw.fif
 - data\2-back\eeg\sub-P001\ses-S001\eeg\sub-P001_ses-S001_task-Default_run-002_eeg_annot_raw.fif
Opening raw data file data\2-back\eeg\sub-P001\ses-S001\eeg\sub-P001_ses-S001_task-Default_run-001_eeg_annot_raw.fif...
    Range : 0 ... 318976 =      0.000 ...   637.952 secs
Ready.
Reading 0 ... 318976  =      0.000 ...   637.952 secs...
Loaded sub-P001_ses-S001_task-Default_run-001_eeg_annot_raw.fif | 20 ch | 638.0 s
Opening raw data file data\2-back\eeg\sub-P001\ses-S001\eeg\sub-P001_ses-S001_task-Default_run-002_eeg_annot_raw.fif...
    Range : 0 ... 598481 =      0.000 ...  1196.962 secs
Ready.
Reading 0 ... 598481  =      0.000 ...  1196.962 secs...
Loaded sub-P001_ses-S001_task-Default_run-002_eeg_annot_raw.fif | 20 ch | 1197.0 s

Concatenated raw:
<Raw | sub-P001_ses-S001_task-Default_run-001_eeg_annot_raw.fif, 20 x 917459 (1834.9 s), ~140.0 MiB, data loaded

C:\Users\n.saleem\AppData\Local\Temp\ipykernel_5468\2861176311.py:70: RuntimeWarning: This filename (c:\Users\n.saleem\Documents\GitHub\0x2x2dual_WCST_analysis\data\2-back\eeg\concatenated_fif.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw_concat.save(out_path, overwrite=True)


Closing c:\Users\n.saleem\Documents\GitHub\0x2x2dual_WCST_analysis\data\2-back\eeg\concatenated_fif.fif
[done]


[WindowsPath('c:/Users/n.saleem/Documents/GitHub/0x2x2dual_WCST_analysis/data/2-back/eeg/concatenated_fif.fif')]